# AEGIS-SQL — Spider-KO 1,034 Full Run

이 노트북은 **Qwen2.5-Coder-1.5B-Instruct**를 기본 모델로 Spider-KO validation 1,034문항을 외부 cross-domain 평가합니다.

**사용법:** Colab에서 GPU 런타임을 선택한 뒤 `런타임 → 모두 실행`을 누르세요. Google Drive 권한 요청이 한 번 표시됩니다. 결과 JSON/ZIP과 provenance manifest는 Drive에 영구 보존됩니다.

> 이 결과는 **공식 Spider leaderboard 점수**가 아닙니다. 프로젝트 내부의 **AEGIS execution_match** 방식으로 측정한 외부 일반화 체크입니다.


In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임이 필요합니다. Colab 런타임 설정에서 T4 GPU를 선택하세요.")

print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
ARCHIVE_DIR = Path("/content/drive/MyDrive/AEGIS-SQL/spider-ko")
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
print("archive:", ARCHIVE_DIR)


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/aegis-sql")
REPO_URL = "https://github.com/sokldjs554/aegis-sql.git"

if not REPO_DIR.exists():
    subprocess.run(f"git clone --branch main {REPO_URL} {REPO_DIR}", shell=True, check=True)
else:
    subprocess.run("git checkout main", cwd=REPO_DIR, shell=True, check=True)
    subprocess.run("git pull --ff-only origin main", cwd=REPO_DIR, shell=True, check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)


In [ ]:
import os
import subprocess

env = os.environ.copy()
env["ARCHIVE_DIR"] = str(ARCHIVE_DIR)
subprocess.run(
    "bash scripts/run_spider_ko_colab.sh",
    shell=True,
    check=True,
    env=env,
)


In [ ]:
import json

result_files = sorted(
    ARCHIVE_DIR.glob("spider-ko-full-*.json"),
    key=lambda p: p.stat().st_mtime,
)
if not result_files:
    raise RuntimeError("Spider-KO full result JSON을 찾을 수 없습니다.")

result_path = result_files[-1]
report = json.loads(result_path.read_text(encoding="utf-8"))

if report.get("items") != 1034:
    raise RuntimeError(f"1,034문항 full run이 아닙니다: items={report.get('items')}")
if report.get("portfolio_evidence_ready") is not True:
    raise RuntimeError("portfolio_evidence_ready=false: row-level evidence와 provenance를 확인하세요.")

latency = report.get("latency_ms", {})
print("result:", result_path)
print(f"EX: {report['correct']}/{report['items']} = {report['execution_accuracy']:.1%}")
print(f"p50/p95 latency: {latency.get('p50')} / {latency.get('p95')} ms")
print("execution failures:", report.get("execution_failures"))
print("portfolio_evidence_ready:", report["portfolio_evidence_ready"])


## 해석 원칙

- 이 수치는 **공식 Spider leaderboard 점수**가 아니라 `AEGIS execution_match` 기반의 외부 cross-domain 일반화 체크입니다.
- KorFin-Bench와 Spider-KO 수치를 하나의 점수처럼 합치지 않습니다.
- 1.5B baseline 결과가 확보되면 먼저 row-level 실패 유형을 분석하고, 그 근거가 있을 때만 3B/8B 모델 스케일업이나 추가 QLoRA 실험을 진행합니다.
